# Exploration notebook

Template for cross-experiment analysis. Edit freely — gitignored after initial commit.
Promote any reusable pattern to `analysis/plot.py`.

In [ ]:
import sys; sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import analysis.plot as P

from analysis.load import load_results, filter_results, to_dataframe, load_scene
from analysis.plot import (metric_vs_param, convergence_curves, compare_algorithms,
                            sanity_check, diff_error_map, sam_map, spectral_profile)

# ── Global switches ────────────────────────────────────────────
P.SAVE_FIGURES = False   # True  → save PNGs to OUTPUT_DIR
                         # False → display inline
STUDY_DIR   = '../results/aggregate'   # ← point here
OUTPUT_DIR  = '../figs/explore'
RGB_INDICES = [20, 10, 5]             # bands for RGB preview
# ──────────────────────────────────────────────────────────────

PARAMS  = ['algorithm', 'lmbda', 'lmbda_m', 'p', 'q', 'r',
           'scale', 'noise_level', 'sigma_blur', 'max_iter', 'max_iter_cp']
METRICS = ['PSNR_mean', 'SSIM_mean', 'SAM_mean', 'RNMSE_mean', 'CC_mean']

results = load_results(STUDY_DIR)
df = to_dataframe(results)
print(f'{len(df)} experiments | {df["algorithm"].value_counts().to_dict()}')

## 1. Coverage — what has been run?

In [ ]:
# Full table, sorted by PSNR
cols = [c for c in PARAMS if c in df.columns] + [c for c in METRICS if c in df.columns]
df[cols].sort_values('PSNR_mean', ascending=False)

In [ ]:
# Parameter space coverage — adjust axes to what you care about
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=df, x='lmbda', y='noise_level',
                hue='algorithm', size='PSNR_mean', sizes=(40, 200), ax=ax)
ax.set_xscale('log')
ax.set_title('Explored parameter space')
plt.tight_layout()

## 2. Define your slice

Set `FILTERS` once. All cells below use `subset` / `subset_df`.
Comment out keys to leave that dimension free.

In [ ]:
# ── Edit here ─────────────────────────────────────────────────
FILTERS = dict(
    noise_level = 40,
    # sigma_blur  = 1.0,
    # max_iter_cp = 50,
)
# ──────────────────────────────────────────────────────────────

subset    = filter_results(results, **FILTERS)
subset_df = to_dataframe(subset)

free = subset_df[[c for c in PARAMS if c in subset_df.columns]]
free_params = free.columns[free.nunique() > 1].tolist()
print(f'{len(subset)} experiments match')
print(f'Free parameters: {free_params}')
subset_df[[c for c in free_params + [m for m in METRICS if m in subset_df.columns]]]

## 3. Sanity check — one experiment

Set `i` to navigate through experiments in the current slice.

In [ ]:
i = 0   # ← 0 … len(subset)-1

r = subset[i]
print({k: r.get(k) for k in free_params + ['algorithm']})

scene = load_scene(r)   # loads GT, LR HSI, PAN from dataset
sanity_check(r, scene, output_dir=OUTPUT_DIR, rgb_indices=RGB_INDICES)

## 4. Spatial comparison — CTV vs GradAlign

Pick matched experiments (same params, different algorithm) and compare spatially.

In [ ]:
# Fix everything except algorithm — take first match of each
fixed = {k: v for k, v in FILTERS.items()}   # add more constraints here if needed
r_ctv = filter_results(results, algorithm='CTV',       **fixed)[0]
r_ga  = filter_results(results, algorithm='GradAlign', **fixed)[0]

scene = load_scene(r_ctv)   # same degradation → same scene for both

diff_error_map(r_ctv['reconstructed'][0], r_ga['reconstructed'][0],
               scene['gt'], scene['ym'],
               label_a='CTV', label_b='GradAlign', output_dir=OUTPUT_DIR)

In [ ]:
# SAM maps side by side
sam_ctv = sam_map(r_ctv['reconstructed'][0], scene['gt'],
                  title='SAM — CTV', output_dir=OUTPUT_DIR)
sam_ga  = sam_map(r_ga['reconstructed'][0],  scene['gt'],
                  title='SAM — GradAlign', output_dir=OUTPUT_DIR)
print(f'Mean SAM  CTV: {sam_ctv.mean():.3f}°   GradAlign: {sam_ga.mean():.3f}°')

In [ ]:
# Spectral profile at a chosen pixel
# Pick a pixel at a PAN edge (high gradient) to test alignment claim
pixel = (120, 80)   # ← (y, x)

spectral_profile(pixel, scene['gt'], output_dir=OUTPUT_DIR,
                 CTV=r_ctv['reconstructed'][0],
                 GradAlign=r_ga['reconstructed'][0])

## 5. Global metrics and convergence

In [ ]:
compare_algorithms({'CTV': filter_results(subset, algorithm='CTV'),
                    'GradAlign': filter_results(subset, algorithm='GradAlign')},
                   'PSNR', output_dir=OUTPUT_DIR)

In [ ]:
metric_vs_param(subset, param='lmbda', metric_name='PSNR', output_dir=OUTPUT_DIR)

In [ ]:
convergence_curves(subset, group_by='lmbda', facet_by='algorithm',
                   mode='distance', output_dir=OUTPUT_DIR)

## 6. Deep dive — free-form

In [ ]:
# Two-parameter interaction
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(subset_df['lmbda'], subset_df['noise_level'],
                c=subset_df['PSNR_mean'], cmap='viridis', s=80)
plt.colorbar(sc, ax=ax, label='PSNR')
ax.set_xscale('log')
ax.set_xlabel('lmbda'); ax.set_ylabel('noise_level')
ax.set_title('PSNR over (lmbda, noise_level)')
plt.tight_layout()